# Tools with MCP ⏰

The Model Context Protocol (MCP) provides a standardized way to connect AI agents to external tools and data sources. Let's connect to an MCP server using `langchain-mcp-adapters`.

## Setup

Load and/or check for needed environmental variables

In [20]:
from dotenv import load_dotenv
from env_utils import doublecheck_env
#检查环境加载API_KEY
# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env(".env")

OPENAI_API_KEY=****ywsA
DEEPSEEK_API_KEY=****c1e4
DEEPSEEK_BASE_URL=****.com
LANGSMITH_API_KEY=****b63c
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [21]:
# from langchain_mcp_adapters.client import MultiServerMCPClient
# import nest_asyncio
#
# """
# Jupyter 里已经有自己的事件循环，
# 所以这里先用 nest_asyncio 让 await / async with 能直接运行。
# 这一步本身没问题。
# """
# nest_asyncio.apply()
#
# # 这里原本想通过 npx 启动 mcp-time MCP 服务器，
# # 再把服务器暴露出来的工具加载成 LangChain tools。
# # 逻辑上没错，但这条路径在当前 Windows 环境里会失效。
# mcp_client = MultiServerMCPClient(
#     {
#         "time": {
#             # stdio 表示通过标准输入/输出和子进程通信
#             "transport": "stdio",
#             # 用 npx 临时拉起 npm 包
#             "command": "npx",
#             # @theo.foobar/mcp-time 是 MCP 时间服务器包
#             "args": ["-y", "@theo.foobar/mcp-time"],
#         }
#     },
# )
#
# # 这里会尝试启动 MCP server 并读取工具列表
# # 但当前会失败，因为这个 npm 包的 postinstall 脚本有问题：
# # 它在安装时硬编码了一个旧版本号 0.1.0，
# # 而当前发布的 0.4.0 包对应的 Windows 二进制并没有 0.1.0 这个版本。
# # 结果就是安装脚本失败，MCP 服务器根本没真正启动，
# # 所以后面会出现 Connection closed / 502 Bad Gateway。
# mcp_tools = await mcp_client.get_tools()
#
# print(f"Loaded {len(mcp_tools)} MCP tools: {[t.name for t in mcp_tools]}")



import os
import socket
import subprocess
import sys
import time
from pathlib import Path

import httpx
import nest_asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

# Jupyter Notebook 自己已经在运行事件循环了
# 如果我们还要在 notebook 里继续使用 await / async 代码，
# 有时会和 notebook 原本的事件循环冲突
# nest_asyncio 的作用就是让这个事件循环支持“嵌套使用”
nest_asyncio.apply()

# 如果上一次运行这个 cell 时，已经启动过一个本地 MCP 服务
# 那么先把旧进程停掉，避免重复启动导致端口冲突
if "mcp_server_process" in globals() and mcp_server_process and mcp_server_process.poll() is None:
    mcp_server_process.terminate()
    try:
        # 等它优雅退出
        mcp_server_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        # 如果 5 秒内还没退出，就强制结束
        mcp_server_process.kill()

# 这里先找一个空闲端口
# 因为我们要在本地启动一个 MCP 服务，
# 不能直接写死端口，写死可能会和别的程序冲突
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind(("127.0.0.1", 0))
    # 0 表示让系统自动分配一个没被占用的端口
    local_port = s.getsockname()[1]

# 这里是本地 MCP 服务脚本的位置
# Path.resolve() 会把相对路径变成绝对路径
server_script = Path("l5_local_mcp_server.py").resolve()

# 准备启动服务时要用的环境变量
# 这里把 MCP_PORT 传给子进程，让服务知道自己要监听哪个端口
server_env = os.environ.copy()
server_env["MCP_PORT"] = str(local_port)

# 启动本地 MCP 服务进程
# 这个服务会在后台单独跑起来
# 我们后面再通过 HTTP 去连接它
mcp_server_process = subprocess.Popen(
    [sys.executable, str(server_script)],
    env=server_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# 启动服务后，先等它真正起来
# 因为刚启动时服务可能还没准备好
# 所以这里循环检查端口是否已经可以连接
deadline = time.time() + 15
while True:
    try:
        with socket.create_connection(("127.0.0.1", local_port), timeout=1):
            # 能连上说明服务已经起来了
            break
    except OSError:
        # 如果超过 15 秒还没起来，就直接报错
        if time.time() > deadline:
            raise RuntimeError(f"Local MCP server did not start on port {local_port}")
        time.sleep(0.2)

# 这里定义一个 httpx 客户端工厂
# trust_env=False 的意思是：
# 不要读取系统代理环境变量
# 这样本地 localhost 请求就不会被代理干扰
def httpx_factory(**kwargs):
    return httpx.AsyncClient(trust_env=False, **kwargs)

# 创建 MCP 客户端
# 这里不是连接外部 npx 服务，而是连接我们刚启动的本地服务
mcp_client = MultiServerMCPClient(
    {
        "math": {
            # 使用 streamable_http 方式连接
            "transport": "streamable_http",
            # 本地 MCP 服务地址
            "url": f"http://127.0.0.1:{local_port}/mcp",
            # 使用我们自定义的 httpx 客户端
            "httpx_client_factory": httpx_factory,
        }
    }
)

# 从 MCP 服务里读取工具列表
# 最终 mcp_tools 就是一组 LangChain 可用的工具对象
mcp_tools = await mcp_client.get_tools()

# 打印加载结果，方便确认是否成功
print(f"Loaded {len(mcp_tools)} MCP tools: {[t.name for t in mcp_tools]}")


Loaded 2 MCP tools: ['add', 'multiply']


Create an agent with the MCP-provided time tools.

In [22]:
from langchain.agents import create_agent

agent_with_mcp = create_agent(
    model="deepseek-chat",
    # model="openai:gpt-5",
    tools=mcp_tools,
    system_prompt="You are a helpful assistant",
)

Ask about the current time in San Francisco.

In [23]:
result = await agent_with_mcp.ainvoke(
    {"messages": [{"role": "user", "content": "使用MCP数学工具计算23与19的和，以及7与8的乘积，并给出最终计算结果。"}]}
)
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

使用MCP数学工具计算23与19的和，以及7与8的乘积，并给出最终计算结果。
================================== Ai Message ==================================

我将使用MCP数学工具来计算这两个表达式。

首先，计算23与19的和：
Tool Calls:
  add (call_00_MUVr2oxhX6DhoaimXcQzcBNd)
 Call ID: call_00_MUVr2oxhX6DhoaimXcQzcBNd
  Args:
    a: 23
    b: 19
================================= Tool Message =================================
Name: add

42
================================== Ai Message ==================================

接下来，计算7与8的乘积：
Tool Calls:
  multiply (call_00_I4WKSuu7SPKsyLGsKasAOniJ)
 Call ID: call_00_I4WKSuu7SPKsyLGsKasAOniJ
  Args:
    a: 7
    b: 8
================================= Tool Message =================================
Name: multiply

56
================================== Ai Message ==================================

最终计算结果：
1. 23与19的和 = 42
2. 7与8的乘积 = 56

所以，两个计算的结果分别是42和56。
